<a href="https://colab.research.google.com/github/martim-p05/detecao-anatomia-abelhas/blob/bee_species_identification/02_training_20_species.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import gc

# 1 Get Google Drive
drive.mount('/content/drive')
print("Google Drive mounted successfully.\n")

# 2 Paths
working_folder = Path('/content/drive/MyDrive/robin_2/')
data_file_path = working_folder / 'temp_files' / 'dataset_20_species.npz'
test_data_file_path = working_folder / 'temp_files' / 'test_data_20_species.npz'
keras_model_file_path = working_folder / 'model' / 'bee_20_species_model.keras'
final_model_path = working_folder / 'model' / 'bee_20_species_model_final.keras'

# 3 Load the dataset created in script 1
print(f"Reading data from: {data_file_path}")
if data_file_path.exists():
    loaded_data = np.load(data_file_path)
    X = loaded_data['X']
    labels = loaded_data['labels']
    print(f"Successfully loaded X with shape: {X.shape}")
else:
    raise FileNotFoundError(f" '{data_file_path}' does not exist.")

# 4. Encode Labels
encoder = LabelEncoder()
y = encoder.fit_transform(labels)
print(f"Number of classes: {len(encoder.classes_)}")

# 5. Split Data and Clear RAM (CRITICAL FOR MEMORY)
print("\nSplitting data into Train and Temp...")
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("Deleting original dataset from memory to free RAM")
del X
del labels
gc.collect()

print("Splitting Temp into Validation and Test...")
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

del X_temp
gc.collect()

# Save Test Data for Script 3
np.savez_compressed(test_data_file_path, X_test=X_test, y_test=y_test)
print(f"Test data saved to {test_data_file_path}\n")


# Clear deep Keras backend memory to ensure a clean slate
tf.keras.backend.clear_session()
gc.collect()

# 6 Load the preprocess model
print(f"Loading the warm-up model from {keras_model_file_path}...")
model = load_model(keras_model_file_path)

# 7 PARTIAL UNFREEZE: Only unlock the top 50 layers to save RAM
print("Unlocking ONLY the top 50 layers for memory-efficient fine-tuning...")
for layer in model.layers:
    layer.trainable = False
for layer in model.layers[-50:]:
    layer.trainable = True


# 0.00001 prevents the AI from forgetting the basic features it already knows
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

# 9 Early Stopping AND Auto-Save (Checkpoint)
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True,
    verbose=1
)


checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=final_model_path,
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

# 10 Start Training (WITH REDUCED BATCH SIZE AND AUTO-SAVE)
print("\nStarting the training (up to 75 epochs) with Auto-Save...")
history_fine = model.fit(
    X_train,
    y_train,
    epochs=75,
    batch_size=16,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, checkpoint],
    verbose=1
)

print(f"\nSUCCESS, training complete. Best model is saved at: {final_model_path}")

Mounted at /content/drive
Google Drive mounted successfully.

Reading data from: /content/drive/MyDrive/robin_2/temp_files/dataset_20_species.npz
Successfully loaded X with shape: (6834, 224, 224, 3)
Number of classes: 20

Splitting data into Train and Temp...
Deleting original dataset from memory to free RAM
Splitting Temp into Validation and Test...
Test data saved to /content/drive/MyDrive/robin_2/temp_files/test_data_20_species.npz

Loading the warm-up model from /content/drive/MyDrive/robin_2/model/bee_20_species_model.keras...
Unlocking ONLY the top 50 layers for memory-efficient fine-tuning...

Starting the training (up to 75 epochs) with Auto-Save...
Epoch 1/75
299/299 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step - accuracy: 0.1589 - loss: 2.7847
Epoch 1: val_accuracy improved from None to 0.26439, saving model to /content/drive/MyDrive/robin_2/model/bee_20_species_model_final.keras

Epoch 1: finished saving model to /content/drive/MyDrive/robin_2/model/bee_20_species_model_final.keras
2